In [16]:
# ---- Imports ----
import time
import re
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

SEARCH_URL = "https://query2.finance.yahoo.com/v1/finance/search"

# ---- Tiny rate limiter (keeps Yahoo happy) ----
class RateLimiter:
    def __init__(self, rps=4):
        self.rps = max(1, int(rps))
        self.tokens = self.rps
        self.last = time.time()
    def acquire(self):
        now = time.time()
        elapsed = now - self.last
        if elapsed >= 1:
            self.tokens = self.rps
            self.last = now
        while self.tokens <= 0:
            time.sleep(0.12)
            now = time.time()
            if now - self.last >= 1:
                self.tokens = self.rps
                self.last = now
        self.tokens -= 1

# ---- Yahoo search: return up to N candidates (symbol, shortname) ----
def yahoo_candidates(query: str, session: requests.Session, rlim: RateLimiter, max_hits=10):
    if not isinstance(query, str) or not query.strip():
        return []
    try:
        rlim.acquire()
        r = session.get(
            SEARCH_URL,
            params={"q": query, "quotesCount": max_hits, "newsCount": 0, "lang": "en-US", "region": "US"},
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=10,
        )
        if r.status_code in (401,403,429,500,502,503,504):
            time.sleep(0.6)
            rlim.acquire()
            r = session.get(
                SEARCH_URL,
                params={"q": query, "quotesCount": max_hits, "newsCount": 0, "lang": "en-US", "region": "US"},
                headers={"User-Agent": "Mozilla/5.0"},
                timeout=10,
            )
        r.raise_for_status()
        quotes = (r.json() or {}).get("quotes", []) or []
        out = []
        for q in quotes:
            sym = q.get("symbol")
            if sym:
                out.append((sym, q.get("shortname") or q.get("longname") or ""))
        return out
    except Exception:
        return []

# ---- Utilities for ranking ----
def yf_suffix(symbol: str) -> str:
    """Return Yahoo suffix (without dot). '' for US/no suffix."""
    if not isinstance(symbol, str) or not symbol:
        return ""
    if "." not in symbol:
        return ""   # US
    return symbol.split(".", 1)[1].upper()

def isin_home_country(isin: str) -> str:
    """Map ISIN prefix to ISO alpha-2 (rough home country)."""
    if not isinstance(isin, str) or len(isin) < 2:
        return ""
    return isin[:2].upper()

def build_suffix_country_map(mapping_csv: str) -> dict:
    m = pd.read_csv(mapping_csv, dtype=str)
    m["Suffix"] = m["Suffix"].fillna("").str.replace(r"^\.", "", regex=True).str.upper()
    m["Country"] = m["Country"].fillna("")
    return dict(zip(m["Suffix"], m["Country"]))

# Per-country board preferences (left = best)
PREFS = {
    "CA": ["TO","V","NE","CN",""],   # Canada
    "US": [""],                      # United States (no suffix)
    "GB": ["L","IL"],                # United Kingdom
    "NL": ["AS"], "FR": ["PA"], "BE": ["BR"], "PT": ["LS"],
    "DE": ["DE","F","BE","DU","SG","MU","HM","HA"],
    "CH": ["SW"],
    "DK": ["CO"], "SE": ["ST"], "NO": ["OL"], "FI": ["HE"], "IS": ["IC"],
    "JP": ["T"],
    "CN": ["SS","SZ"],
    "HK": ["HK"],
    "TW": ["TW","TWO"],
    "SG": ["SI"],
    "AU": ["AX"],
    "NZ": ["NZ"],
    "IN": ["NS","BO"],
    "ES": ["MC","BC","BI","MA"],
    "IT": ["MI"], "AT": ["VI"],
    "TR": ["IS"], "GR": ["AT"], "IL": ["TA"],
    "ZA": ["JO"],
    "BR": ["SA"], "MX": ["MX"], "AR": ["BA"], "CL": ["SN"], "PE": ["LM"], "CO": ["COLO"],
    "KR": ["KS","KQ"],
    "TH": ["BK"], "MY": ["KL"], "ID": ["JK"], "PH": ["PS"],
    "VN": ["VN","HN"],
    "AE": ["AD","DU"], "QA": ["QA"], "SA": ["SR"], "BH": ["BH"], "KW": ["KW"], "OM": ["MS"],
    "IE": ["IR"],
}

def pick_best_symbol(isin: str, name: str, candidates, suffix_country_map: dict):
    """Rank Yahoo candidates:
       1) suffix-country == ISIN home-country (by prefix)
       2) board preference in that country
       3) symbol simplicity (fewer punctuation, shorter)
    """
    home2 = isin_home_country(isin)        # e.g., 'CA'
    pref  = PREFS.get(home2, [])
    def country_of(sym):
        suf = yf_suffix(sym)
        if suf == "":
            return "United States"  # US/no suffix
        return suffix_country_map.get(suf, "")

    def score(sym):
        ctry = country_of(sym)
        suf  = yf_suffix(sym)
        # normalize ISO alpha-2 to country bucket in PREFS
        alpha2_to_country = {
            "CA":"Canada","US":"United States","GB":"United Kingdom","NL":"Netherlands","FR":"France","BE":"Belgium",
            "PT":"Portugal","DE":"Germany","CH":"Switzerland","DK":"Denmark","SE":"Sweden","NO":"Norway","FI":"Finland","IS":"Iceland",
            "JP":"Japan","CN":"China","HK":"Hong Kong","TW":"Taiwan","SG":"Singapore","AU":"Australia","NZ":"New Zealand",
            "IN":"India","ES":"Spain","IT":"Italy","AT":"Austria","TR":"Turkey","GR":"Greece","IL":"Israel","ZA":"South Africa",
            "BR":"Brazil","MX":"Mexico","AR":"Argentina","CL":"Chile","PE":"Peru","CO":"Colombia","KR":"South Korea","TH":"Thailand",
            "MY":"Malaysia","ID":"Indonesia","PH":"Philippines","VN":"Vietnam","AE":"UAE","QA":"Qatar","SA":"Saudi Arabia",
            "BH":"Bahrain","KW":"Kuwait","OM":"Oman","IE":"Ireland"
        }
        home_country_name = alpha2_to_country.get(home2, "")
        same_country = 0 if (ctry == home_country_name and ctry != "") else 1
        try:
            board_rank = pref.index(suf) if same_country == 0 else 99
        except ValueError:
            board_rank = 98 if same_country == 0 else 99
        nonword = sum(1 for ch in sym if not ch.isalnum())
        return (same_country, board_rank, nonword, len(sym))

    if not candidates:
        return None
    return sorted(candidates, key=lambda t: score(t[0]))[0][0]


In [ ]:
# === paths (edit these) ===


input_csv  = r"D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned_final.csv"   # change path if needed
output_csv = r"D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_isin_to_ticker.csv" 
mapping_csv = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\yahoo_suffix_mapping_full.csv"         # the suffix→exchange/country CSV
unresolved_csv = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\UniSuper_unresolved_isins.csv"

name_col = "Name/Kind of Investment Item"
isin_col = "Stock ID"

MAX_WORKERS = 10   # threads
RPS         = 5    # requests per second total (raise cautiously)

# Load & keep ONLY non-empty Stock IDs
df_src = pd.read_csv(input_csv)
mask_nonempty = df_src[isin_col].notna() & df_src[isin_col].astype(str).str.strip().ne("")
df = df_src.loc[mask_nonempty, [isin_col, name_col]].copy()

# Build suffix→country map once
suffix_country_map = build_suffix_country_map(mapping_csv)

# --- Phase 1: ISIN queries only (dedup) ---
isins = df[isin_col].astype(str).tolist()
unique_isins = sorted({i for i in isins if i.strip()})
cands_by_isin = {}

rlim = RateLimiter(rps=RPS)
with requests.Session() as session, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(yahoo_candidates, q, session, rlim, max_hits=3): q for q in unique_isins}
    done = 0
    total = len(futures)
    for fut in as_completed(futures):
        q = futures[fut]
        cands_by_isin[q] = fut.result()
        done += 1
        if done % 200 == 0 or done == total:
            print(f"[ISIN] {done}/{total}")

# pick best from ISIN pass
symbols = []
unresolved_idx = []
names = df[name_col].astype(str).tolist()
for idx, (isin, nm) in enumerate(zip(isins, names)):
    cands = cands_by_isin.get(isin, [])
    best = pick_best_symbol(isin, nm, cands, suffix_country_map) if cands else None
    symbols.append(best)
    if best is None:
        unresolved_idx.append(idx)

df["YF_Symbol"] = symbols

# --- Phase 2: Name queries only for unresolved rows (dedup) ---
if unresolved_idx:
    unresolved_names = [names[i] for i in unresolved_idx]
    unique_names = sorted({n for n in unresolved_names if n.strip()})
    cands_by_name = {}

    with requests.Session() as session, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(yahoo_candidates, q, session, rlim, max_hits=3): q for q in unique_names}
        done = 0
        total = len(futures)
        for fut in as_completed(futures):
            q = futures[fut]
            cands_by_name[q] = fut.result()
            done += 1
            if done % 200 == 0 or done == total:
                print(f"[NAME] {done}/{total}")

    # fill unresolved with best name-based pick
    for idx in unresolved_idx:
        isin = isins[idx]
        nm   = names[idx]
        cands = cands_by_name.get(nm, [])
        best = pick_best_symbol(isin, nm, cands, suffix_country_map) if cands else None
        df.at[idx, "YF_Symbol"] = best

# Suffix only when a real symbol exists
def suffix_or_na(sym):
    if not isinstance(sym, str) or not sym.strip():
        return pd.NA
    return yf_suffix(sym)
df["YF_Suffix"] = df["YF_Symbol"].apply(suffix_or_na)

# Merge suffix → Exchange/Country (only for rows with a symbol)
map_df = pd.read_csv(mapping_csv, dtype=str)
map_df["Suffix"] = map_df["Suffix"].fillna("").str.replace(r"^\\.", "", regex=True).str.upper()
map_df = map_df.rename(columns={"Suffix":"_join_suffix", "Exchange":"YF_Exchange", "Country":"YF_Country"})

df["_join_suffix"] = pd.NA
has_symbol = df["YF_Symbol"].notna() & df["YF_Symbol"].astype(str).str.strip().ne("")
df.loc[has_symbol, "_join_suffix"] = (
    df.loc[has_symbol, "YF_Suffix"]
      .astype("string")
      .str.replace(r"^\\.", "", regex=True)
      .str.upper()
)
df_enriched = df.merge(map_df, how="left", on="_join_suffix")

# Split resolved vs unresolved
resolved_mask = has_symbol
out_resolved = df_enriched[[isin_col, name_col, "YF_Symbol", "YF_Suffix", "YF_Exchange", "YF_Country"]][resolved_mask].copy()
out_unresolved = df_enriched[[isin_col, name_col]].loc[~resolved_mask].copy()

# Write files
out_resolved.to_csv(output_csv, index=False)
out_unresolved.to_csv(unresolved_csv, index=False)
print(f"✔ Saved {len(out_resolved)} resolved rows to {output_csv}")
print(f"✔ Saved {len(out_unresolved)} unresolved rows to {unresolved_csv}")




Progress: 200/6470
Progress: 400/6470
Progress: 600/6470
Progress: 800/6470
Progress: 1000/6470
Progress: 1200/6470
Progress: 1400/6470
Progress: 1600/6470
Progress: 1800/6470
Progress: 2000/6470
Progress: 2200/6470
Progress: 2400/6470
Progress: 2600/6470
Progress: 2800/6470
Progress: 3000/6470
Progress: 3200/6470
Progress: 3400/6470
Progress: 3600/6470
Progress: 3800/6470
Progress: 4000/6470
Progress: 4200/6470
